# 03: Supervisor e o grafo do pipeline

Aqui os agentes dos notebooks 01 e 02 viram um **pipeline**: um `StateGraph` do LangGraph em que o
**Supervisor** decide quem trabalha a cada passo e todos os agentes voltam para ele.

No estudo inicial do padrão, o supervisor era um agente que escolhia tools sozinho. Aqui ele é uma **função Python**:
regras objetivas rodam em código (os *guards*) e só o julgamento que precisa de contexto vai para o LLM.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. O desenho do grafo

Nós: `Supervisor`, os três workers (`YFinance`, `Analyzer`, `Investigator`) e o `Consolidador`. As setas
pontilhadas saem do Supervisor (ele escolhe o próximo); as cheias voltam a ele. Do Consolidador, o fluxo vai ao fim.

In [2]:
from fluxo_financeiro import app

print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	Supervisor(Supervisor)
	YFinance(YFinance)
	Analyzer(Analyzer)
	Investigator(Investigator)
	Consolidador(Consolidador)
	__end__([<p>__end__</p>]):::last
	Analyzer --> Supervisor;
	Investigator --> Supervisor;
	Supervisor -.-> Analyzer;
	Supervisor -.-> Consolidador;
	Supervisor -.-> Investigator;
	Supervisor -.-> YFinance;
	YFinance --> Supervisor;
	__start__ --> Supervisor;
	Consolidador --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. O estado que circula

O grafo carrega um dicionário (`MessagesState`) que cada nó lê e atualiza:

| Campo | O que guarda |
|---|---|
| `messages` | o histórico; cada mensagem de agente tem `name=` dizendo quem falou |
| `next_agent` | a decisão do Supervisor |
| `loop_count` | quantas vezes o Supervisor passou (soma 1 por passagem) |
| `decidido_por` | quem decidiu: `guard`, `llm` ou `hard_stop` |
| `supervisor_reasoning` | a justificativa da decisão |
| `sinais_anomalia` | sinais calculados em código pelo nó do YFinance (notebook 10) |
| `relatorio` | o relatório estruturado do Consolidador (notebook 11) |

## 3. O pipeline de 4 etapas

| Etapa | Situação | Destino | Quem decide |
|---|---|---|---|
| 1 | Ainda não há dados | YFinance | LLM |
| 2 | YFinance coletou, Analyzer não rodou | Analyzer (empresa) | guard |
| 3 | Analyzer falou uma vez | Analyzer (setor) | guard |
| 4 | Analyzer falou duas vezes e há sinais de anomalia | Investigator | guard |
| 4 | Analyzer falou duas vezes, sem sinais | Investigator ou FINISH | LLM |
| fim | Investigator já respondeu | FINISH | guard |
| fim | `loop_count` chegou a 6 | FINISH | hard stop |

Todo FINISH passa pelo **Consolidador** antes de acabar: quem manda para lá é o `router`, não o LLM.
Na etapa 4 sem sinais, o LLM responde a pergunta do prompt: *"Um investidor ficaria com dúvida de POR QUE
esse número aconteceu?"*

In [3]:
from fluxo_financeiro.graph.supervisor import SUPERVISOR_PROMPT

inicio = SUPERVISOR_PROMPT.index("<pipeline>")
fim = SUPERVISOR_PROMPT.index("</pipeline>") + len("</pipeline>")
print(SUPERVISOR_PROMPT[inicio:fim])

<pipeline>
Siga esta ordem rigorosamente:
Etapa 1 (dados financeiros): se não há mensagem do YFinance, rotear para 'YFinance'.
Etapa 2 (notícias da empresa): se há YFinance mas nenhuma mensagem do Analyzer, rotear para 'Analyzer' (já tratado automaticamente pelo guard).
Etapa 3 (contexto do setor): se há exatamente 1 mensagem do Analyzer, 'Analyzer' com tarefa sobre o setor (já tratado automaticamente pelo guard).
Etapa 4 (investigação condicional): se já há 2 ou mais mensagens do Analyzer:
  Se a mensagem do YFinance trouxer "SINAIS DE ANOMALIA", essa etapa já é tratada automaticamente pelo guard.
  Você só decide aqui quando não há sinais calculados; mesmo assim pode haver algo que mereça investigação.
  Leia o contexto completo e pergunte: "Um investidor ficaria com dúvida de POR QUE esse número aconteceu?"
  Se SIM: next_agent 'Investigator', task_for_agent com a descrição precisa da anomalia e o que investigar.
  Se NÃO: next_agent 'FINISH'.
</pipeline>


## 4. Uma análise completa, passo a passo (WEGE3)

`app.stream` devolve a atualização de cada nó assim que ele termina. Imprimimos, para cada passo, quem
rodou, o que o Supervisor decidiu e quem decidiu. Esta célula faz a análise inteira de verdade
(cerca de 20 a 30 segundos e alguns centavos de dólar).

In [4]:
from langchain_core.messages import HumanMessage

passos = []
final = None
for evento in app.stream({"messages": [HumanMessage("Analise a ação WEGE3")]}):
    for no, update in evento.items():
        update = update or {}
        if no == "Supervisor":
            print(f"[Supervisor] -> {update.get('next_agent')}  (decidido por: {update.get('decidido_por')})")
            if update.get("supervisor_reasoning"):
                print(f"   motivo: {update['supervisor_reasoning'][:160]}")
            passos.append(f"Supervisor->{update.get('next_agent')}")
        else:
            texto = update["messages"][-1].text if update.get("messages") else ""
            print(f"[{no}] {len(texto)} caracteres: {texto[:160]!r}")
            passos.append(no)
            if no == "Consolidador":
                final = texto

print("\nCaminho:", " | ".join(passos))

2026-11-08 13:19:51,556 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 13:19:52,740 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, indicadores fundamentalistas e métricas da WEGE3.'


2026-11-08 13:19:52,741 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 13:19:52,742 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


[Supervisor] -> YFinance  (decidido por: llm)
   motivo: Iniciando o pipeline para o ticker WEGE3. O primeiro passo obrigatório é a coleta de dados financeiros.


2026-11-08 13:19:53,605 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


... (28 outputs left out)

2026-11-08 13:20:12,655 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (7 msgs)


[Supervisor] -> FINISH  (decidido por: llm)
   motivo: Os dados financeiros, as notícias corporativas e o contexto setorial da WEGE3 já foram devidamente coletados e analisados pelos agentes YFinance e Analyzer. Com


2026-11-08 13:20:15,027 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1769 chars)


[Consolidador] 1769 caracteres: '# Relatório: WEGE3\n\n## Resumo\nA WEG (WEGE3) apresenta fundamentos sólidos com ROE de 29,9%, P/L de 32,3x e lucro líquido de R$ 6,38B, destacando-se como líder e'

Caminho: Supervisor->YFinance | YFinance | Supervisor->Analyzer | Analyzer | Supervisor->Analyzer | Analyzer | Supervisor->FINISH | Consolidador


## 5. A resposta final

A última mensagem do histórico é o relatório em markdown do Consolidador.

In [5]:
from IPython.display import Markdown

Markdown(final)

# Relatório: WEGE3

## Resumo
A WEG (WEGE3) apresenta fundamentos sólidos com ROE de 29,9%, P/L de 32,3x e lucro líquido de R$ 6,38B, destacando-se como líder em seu setor na B3. No entanto, a companhia enfrenta pressões inflacionárias de custos com matérias-primas e tarifas, além de um aumento na dívida corrente e queda na margem bruta.

## Indicadores
- Preço Atual: R$ 48.19
- Variação diária: -0.23%
- Faixa de 52 semanas: R$ 41.15 a R$ 54.53
- Valor de mercado: R$ 210.0 bi
- LPA: R$ 1.49
- P/L: 32.3x
- ROE: 29.9%
- DY: 3.22%
- Sharpe Ratio: 0.18
- Sortino Ratio: 0.28
- Max Drawdown: -22.5%
- VaR 95% (diário): -3.14%
- Beta (vs IBOV): 0.29
- Alpha (anual): 6.6%
- Piotroski F-Score: 5/9
- Net Income: R$ 6.38B
- Operating Cash Flow: R$ 6.45B
- Total Assets cresceu: +2.8%
- Dívida corrente aumentou: +12.5%
- Current Ratio: 1.55 vs 1.76
- Gross Margin: 33.5% vs 33.7%
- Receita líquida trimestral: R$ 10,86 bilhões
- Lucro líquido trimestral: R$ 1,67 bilhão

## Notícias e setor
A WEG é uma companhia de equipamentos eletroeletrônicos líder no subsetor de Máquinas e Equipamentos da B3. Recentemente, a empresa anunciou a aquisição de um terreno na Índia para expandir sua capacidade fabril. A companhia lida com resultados mistos marcados por pressões pontuais de custos, encarecimento superior a 40% no preço do cobre, aumento de cerca de 20% no aço e tarifas de importação nos Estados Unidos, mas mantém forte diversificação geográfica e receitas externas.

## Riscos
- Dívida corrente aumentou (+12.5%)
- Current Ratio piorou (1.55 vs 1.76)
- Gross Margin caiu (33.5% vs 33.7%)
- Pressões pontuais de custos e tarifas nas margens
- Encarecimento superior a 40% no preço do cobre
- Aumento de cerca de 20% no aço
- Tarifas de importação nos Estados Unidos


## O que observar

- No caminho da WEGE3, a etapa 4 normalmente termina em FINISH sem Investigator: sem sinais de anomalia, o LLM não viu um "porquê" a investigar. A VALE3 tem sinais, e lá o Investigator entra por guard (notebook 10).
- Os passos 2 e 3 dizem `decidido por: guard`: nenhuma chamada ao LLM do Supervisor. O primeiro e o último dizem `llm`.
- O Consolidador não aparece como decisão do Supervisor: o `router` manda todo FINISH para ele.
- São 4 passagens do Supervisor numa análise sem Investigator e 5 com ele; o hard stop em 6 é só margem de segurança.